In [ ]:
%run ./utils_common

In [ ]:
import time
from datetime import date
from decimal import Decimal
from typing import Any, Dict, List, Optional, Tuple

import boto3
from botocore.exceptions import ClientError

In [ ]:
logger = setup_logger('AWSCostExplorer')
logging.getLogger('boto3').setLevel(logging.WARNING)
logging.getLogger('botocore').setLevel(logging.WARNING)

In [ ]:
dbutils.widgets.text('catalog', '', 'CATALOG')
dbutils.widgets.text('schema', '', 'SCHEMA')
dbutils.widgets.text('overlap_days', '3', 'Overlap days (min 2)')

In [ ]:
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
overlap_days = get_overlap_days(dbutils.widgets.get('overlap_days'), logger=logger)

In [ ]:
# =======================================================
# AWS Cost Classification Framework
# =======================================================

# Single source of truth for AWS service-to-category mapping.
# Add new services here to classify them. Any service NOT in this
# map is routed to "other" (never silently assigned to compute).
AWS_SERVICE_CATEGORIES: Dict[str, str] = {
  'Amazon Elastic Compute Cloud - Compute': 'compute',
  'EC2 - Other': 'compute',
  'Amazon Elastic Block Store': 'storage',
  'Amazon Simple Storage Service': 'storage',
  'Elastic Load Balancing': 'network',
  'AWS Data Transfer': 'network',
  'Amazon Virtual Private Cloud': 'network',
}

VALID_COST_CATEGORIES = {'compute', 'storage', 'network', 'other'}


def build_aws_category_column():
  """Build a PySpark Column that classifies service_name using AWS_SERVICE_CATEGORIES.

  Uses F.create_map so the dict is the sole source of truth.
  Unknown services map to 'other' -- never silently to 'compute'.
  """
  from itertools import chain

  mapping_pairs = list(chain.from_iterable(AWS_SERVICE_CATEGORIES.items()))
  mapping_expr = F.create_map(*[F.lit(x) for x in mapping_pairs])
  return F.coalesce(mapping_expr[F.col('service_name')], F.lit('other'))


# =======================================================
# AWS Cost Client
# =======================================================


class AWSCostClient:
  """Client for querying AWS Cost Explorer API for Databricks cluster costs.

  Uses Databricks service credentials to assume an IAM role with
  ce:GetCostAndUsage permissions. The Cost Explorer API endpoint
  is only available in us-east-1, regardless of where resources run.

  Queries costs grouped by ClusterId tag AND SERVICE dimension,
  producing per-cluster daily costs segmented into compute, storage,
  and network categories.
  """

  CE_REGION = 'us-east-1'

  DEFAULT_SERVICES = [
    'Amazon Elastic Compute Cloud - Compute',
    'EC2 - Other',  # EC2 ancillary costs (Elastic IPs, NAT Gateway, data transfer)
    'Amazon Elastic Block Store',
    'Amazon Simple Storage Service',
    'Elastic Load Balancing',
    'AWS Data Transfer',
    'Amazon Virtual Private Cloud',
  ]

  MAX_CHUNK_DAYS = 30
  MAX_RETRIES = 5
  BASE_RETRY_DELAY = 5

  def __init__(self, service_credential_name: str = 'dbspend-read-ce'):
    session = boto3.Session(
      botocore_session=dbutils.credentials.getServiceCredentialsProvider(service_credential_name),
      region_name=self.CE_REGION,
    )
    self.client = session.client('ce')

  # -------- Public API --------

  def get_cluster_costs_daily(
    self,
    start_date: date,
    end_date: date,
    tag_key: str = 'ClusterId',
    services: Optional[List[str]] = None,
    metric: str = 'AmortizedCost',
  ):
    """Query CE for per-tag daily costs with per-service detail.

    Uses dual GroupBy (TAG + SERVICE dimension) to get cost segmentation
    in a single API call. Returns per-service rows that the caller
    aggregates into compute/storage/network categories.

    `tag_key` selects which Databricks resource tag to group on:
      * 'ClusterId' (default) — per-cluster cost; output rows have
        `cluster_id` populated and `instance_pool_id` NULL.
      * 'DatabricksInstancePoolId' — per-pool cost; output rows have
        `instance_pool_id` populated and `cluster_id` NULL.

    Both shapes have identical columns so the two passes can be unioned
    by the caller. CE allows only 2 GroupBy keys per request, so the
    cluster and pool dimensions cannot be captured in a single call —
    see `docs/plans/shared_clusters_and_pools/05-slice-3-instance-pools.md`.

    Returns:
        Spark DataFrame with columns:
          cluster_id, instance_pool_id, service_name, cost, currency,
          cost_incurred_date
        — or None if no cost data found.
    """
    if services is None:
      services = self.DEFAULT_SERVICES

    chunks = self._build_chunks(start_date, end_date)
    all_rows: List[Dict[str, Any]] = []

    for i, (chunk_start, chunk_end) in enumerate(chunks):
      logger.info(
        f'Querying CE chunk {i + 1}/{len(chunks)} [tag={tag_key}]: {chunk_start} → {chunk_end}'
      )
      rows = self._query_with_retries(chunk_start, chunk_end, tag_key, services, metric)
      all_rows.extend(rows)
      if len(chunks) > 1:
        time.sleep(1)

    if not all_rows:
      return None

    spark_df = self._rows_to_spark_df(all_rows)
    return self._route_tag_value_column(spark_df, tag_key)

  def _route_tag_value_column(self, spark_df, tag_key: str):
    """Rename the parsed tag-value column based on which tag was queried.

    `_parse_response` always emits the tag value as `cluster_id`. This
    method preserves that for the cluster-tag pass and renames to
    `instance_pool_id` for the pool-tag pass, adding the other column
    as NULL so both call shapes share an identical schema.
    """
    if tag_key == 'ClusterId':
      return spark_df.withColumn('instance_pool_id', F.lit(None).cast('string'))
    if tag_key == 'DatabricksInstancePoolId':
      return spark_df.withColumnRenamed('cluster_id', 'instance_pool_id').withColumn(
        'cluster_id', F.lit(None).cast('string')
      )
    raise ValueError(
      f'Unsupported tag_key {tag_key!r}; expected ClusterId or DatabricksInstancePoolId.'
    )

  # -------- Chunking --------

  def _build_chunks(self, start: date, end: date) -> List[Tuple[date, date]]:
    chunks = []
    current = start
    while current <= end:
      chunk_end = min(current + timedelta(days=self.MAX_CHUNK_DAYS - 1), end)
      chunks.append((current, chunk_end))
      current = chunk_end + timedelta(days=1)
    return chunks

  # -------- CE params construction --------

  def _build_ce_params(
    self,
    start: date,
    end: date,
    tag_key: str,
    services: List[str],
    metric: str,
  ) -> dict:
    """Build GetCostAndUsage request parameters.

    Uses dual GroupBy: TAG (ClusterId) + DIMENSION (SERVICE) so we
    get per-service costs per cluster in a single API call.
    CE TimePeriod.End is exclusive, so we add 1 day.
    """
    return {
      'TimePeriod': {
        'Start': start.isoformat(),
        'End': (end + timedelta(days=1)).isoformat(),
      },
      'Granularity': 'DAILY',
      'Metrics': [metric],
      'GroupBy': [
        {'Type': 'TAG', 'Key': tag_key},
        {'Type': 'DIMENSION', 'Key': 'SERVICE'},
      ],
      'Filter': {'Dimensions': {'Key': 'SERVICE', 'Values': services}},
    }

  # -------- Retry logic --------

  def _query_with_retries(
    self,
    start: date,
    end: date,
    tag_key: str,
    services: List[str],
    metric: str,
  ) -> List[Dict[str, Any]]:
    """Execute a CE query with exponential-backoff retries.

    Handles AWS CE rate limiting (LimitExceededException) with
    progressively longer delays, and retries transient errors.
    """
    params = self._build_ce_params(start, end, tag_key, services, metric)
    last_exception = None

    for attempt in range(self.MAX_RETRIES):
      try:
        return self._execute_paginated_query(params, metric)
      except ClientError as e:
        last_exception = e
        error_code = e.response['Error']['Code']

        if error_code == 'LimitExceededException':
          wait = min(self.BASE_RETRY_DELAY * (2**attempt), 120)
          logger.warning(
            f'Rate limited (attempt {attempt + 1}/{self.MAX_RETRIES}), waiting {wait}s'
          )
          time.sleep(wait)
        elif attempt < self.MAX_RETRIES - 1:
          wait = 2**attempt
          logger.warning(
            f'ClientError {error_code} (attempt {attempt + 1}), retrying in {wait}s: {e}'
          )
          time.sleep(wait)
        else:
          raise
      except Exception as e:
        last_exception = e
        if attempt < self.MAX_RETRIES - 1:
          wait = 2**attempt
          logger.warning(f'Unexpected error (attempt {attempt + 1}), retrying in {wait}s: {e}')
          time.sleep(wait)
        else:
          raise

    raise last_exception

  # -------- Pagination --------

  def _execute_paginated_query(self, params: dict, metric: str) -> List[Dict[str, Any]]:
    """Execute a CE query, following pagination tokens to completion."""
    rows: List[Dict[str, Any]] = []
    request_params = params.copy()

    while True:
      response = self.client.get_cost_and_usage(**request_params)
      rows.extend(self._parse_response(response, metric))

      next_token = response.get('NextPageToken')
      if not next_token:
        break

      request_params['NextPageToken'] = next_token
      time.sleep(0.5)

    return rows

  # -------- Response parsing --------

  def _parse_response(self, response: dict, metric: str) -> List[Dict[str, Any]]:
    """Parse a single CE response page into flat row dicts.

    With dual GroupBy, each group has two Keys:
      keys[0] = "TagKey$TagValue" (e.g. "ClusterId$0101-abcdef")
      keys[1] = SERVICE dimension value (e.g. "Amazon Elastic Compute Cloud - Compute")
    """
    rows = []
    for time_block in response.get('ResultsByTime', []):
      period_date = time_block['TimePeriod']['Start']

      for group in time_block.get('Groups', []):
        keys = group.get('Keys', [])
        if len(keys) < 2:
          continue

        raw_tag = keys[0]
        cluster_id = raw_tag.split('$')[-1] if '$' in raw_tag else raw_tag
        service_name = keys[1]

        metric_data = group['Metrics'].get(metric, {})
        amount = float(Decimal(metric_data.get('Amount', '0')))
        currency = metric_data.get('Unit', 'USD')

        if amount == 0.0:
          continue

        rows.append(
          {
            'cluster_id': cluster_id,
            'service_name': service_name,
            'cost': amount,
            'currency': currency,
            'cost_incurred_date': period_date,
          }
        )

    return rows

  # -------- Spark conversion --------

  def _rows_to_spark_df(self, rows: List[Dict[str, Any]]):
    """Convert parsed rows to a Spark DataFrame with proper date types."""
    df = spark.createDataFrame(rows)
    return df.withColumn(
      'cost_incurred_date',
      F.to_date(F.col('cost_incurred_date'), 'yyyy-MM-dd'),
    )

In [ ]:
# =======================================================
# APP
# =======================================================
class AWSCostReporterApp:
  """Orchestrates incremental AWS cost ingestion into the cloud cost table.

  Reads the audit log to determine the last successful run, queries
  AWS Cost Explorer for the incremental window (with overlap for
  idempotent MERGE), classifies costs into compute/storage/network/other,
  and upserts results into the target table.

  Invariant enforced: cloud_cost = compute_cost + storage_cost + network_cost + other_cost
  """

  TABLE_NAME = 'dbspend360_cloud_cost_explorer'

  def __init__(self, catalog, schema, overlap_days, logger):
    self.overlap_days = overlap_days
    self.logger = logger
    self.audit_table = build_table_fqn(catalog, schema, 'dbspend360_audit_log')
    self.target_table = build_table_fqn(catalog, schema, 'dbspend360_cloud_cost_explorer')
    self.error_log_table = build_table_fqn(catalog, schema, 'dbspend360_error_log')
    self.breakdown_table = build_table_fqn(catalog, schema, 'dbspend360_other_cost_breakdown')
    self.client = AWSCostClient()

  def run(self):
    start_dt = end_dt = datetime.now(timezone.utc).date()
    try:
      start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

      self.logger.info(
        f'Querying AWS CE cost from {start_dt} to {end_dt} (overlap_days={self.overlap_days})'
      )

      valid, msg = validate_date_window(start_dt, end_dt, self.overlap_days)
      if not valid:
        raise DataQualityError(msg)

      ensure_cost_columns(self.target_table, logger=self.logger)

      # Two parallel CE queries — the cluster-tag pass populates cluster_id
      # rows; the pool-tag pass populates instance_pool_id rows. CE caps
      # GroupBy at 2 keys (TAG + SERVICE), so both dimensions cannot be
      # collected in a single call. Doubles per-run CE quota use; existing
      # LimitExceededException backoff handles throttling.
      cluster_df = self.client.get_cluster_costs_daily(
        start_date=start_dt,
        end_date=end_dt,
        tag_key='ClusterId',
      )
      pool_df = self.client.get_cluster_costs_daily(
        start_date=start_dt,
        end_date=end_dt,
        tag_key='DatabricksInstancePoolId',
      )

      if cluster_df is None and pool_df is None:
        spark_df = None
      elif pool_df is None:
        spark_df = cluster_df
      elif cluster_df is None:
        spark_df = pool_df
      else:
        spark_df = cluster_df.unionByName(pool_df)

      quality_msg = (
        f'overlap_days={self.overlap_days}, tag_passes=[ClusterId,DatabricksInstancePoolId]'
      )

      if spark_df is None or spark_df.limit(1).count() == 0:
        self.logger.info('No AWS cost data returned for the requested range.')
        merged_row_count = 0
      else:
        inc_df = filter_valid_cost_rows(spark_df)

        if inc_df.limit(1).count() == 0:
          self.logger.info('No rows after filtering by cluster_id and cost_incurred_date.')
          merged_row_count = 0
        else:
          self._log_unclassified_services(inc_df)

          classified = inc_df.withColumn('category', build_aws_category_column())
          write_other_cost_breakdown(
            classified,
            'service_name',
            'AWS',
            self.breakdown_table,
            logger=self.logger,
          )

          agg_df = safe_cache(aggregate_costs_by_category(classified))
          merged_row_count = agg_df.count()

          validate_source_schema(
            agg_df,
            {
              'cluster_id': 'string',
              'currency': 'string',
              'cost_incurred_date': 'date',
              'cloud_cost': 'double',
            },
            self.target_table,
            self.logger,
          )
          validate_no_negative_costs(
            agg_df,
            ['cloud_cost', 'compute_cost', 'storage_cost', 'network_cost', 'other_cost'],
            self.target_table,
            self.logger,
          )
          validate_currency_consistency(agg_df, 'currency', self.target_table, self.logger)

          quality_msg = compute_quality_metrics(
            agg_df,
            merged_row_count,
            self.overlap_days,
            logger=self.logger,
          )

          merge_cloud_cost_explorer(self.target_table, agg_df)
          safe_unpersist(agg_df)

          merge_metrics = get_merge_metrics(self.target_table, self.logger)
          quality_msg += (
            f', merge_inserted={merge_metrics.get("num_inserted", "?")}'
            f', merge_updated={merge_metrics.get("num_updated", "?")}'
          )

          validate_post_merge(
            self.target_table,
            'cost_incurred_date',
            start_dt,
            end_dt,
            merged_row_count,
            self.logger,
          )

      self.logger.info(
        f'Merged {merged_row_count} rows into {self.target_table} '
        f'for {start_dt} → {end_dt} (overlap_days={self.overlap_days}).'
      )

      log_audit_run(
        self.audit_table,
        self.TABLE_NAME,
        start_dt,
        end_dt,
        'SUCCESS',
        merged_row_count,
        quality_msg,
      )

    except Exception as e:
      msg = str(e)[:1000]
      self.logger.error(f'Run failed: {msg}')
      try:
        log_audit_run(
          self.audit_table,
          self.TABLE_NAME,
          start_dt,
          end_dt,
          'FAILED',
          0,
          msg,
        )
      except Exception:
        self.logger.error('Failed to write FAILED audit entry')
      raise

  def _log_unclassified_services(self, df):
    """Log unclassified services to both logger and error_log table."""
    known = set(AWS_SERVICE_CATEGORIES.keys())
    svc_costs = (
      df.groupBy('service_name')
      .agg(
        F.sum('cost').alias('total_cost'),
        F.count('*').alias('row_count'),
      )
      .collect()
    )
    unknown_rows = [r for r in svc_costs if r.service_name not in known]

    if not unknown_rows:
      return

    svc_names = [r.service_name for r in unknown_rows]
    self.logger.warning(f'Unclassified AWS services (routed to other_cost): {svc_names}')

    try:
      error_details = [
        f'Unclassified service: {r.service_name}, total_cost=${r.total_cost:.4f}, rows={r.row_count}'
        for r in unknown_rows
      ]
      write_error_log_entries(error_details, 'AWS', 'UNCLASSIFIED_COST', self.error_log_table)
    except Exception as e:
      self.logger.warning(f'Failed to write unclassified services to error_log: {e}')

In [ ]:
# =======================================================
# Execute
# =======================================================
app = AWSCostReporterApp(catalog, schema, overlap_days, logger)
app.run()